# Video Streaming: Feature Extraction from Video Streams

In this assignment, you will explore a capture of a Netflix video stream. The packet capture itself has some additional traffic beyond Netflix traffic, and so part of the exercise involves filtering the traffic to include only the Netflix traffic.

## Learning Objectives

In this hands-on activity, you will learn how to:

* Identify service types using DNS
* Calculate network counters
* Infer video segment downloads


## Step 1: Identifying Netflix Traffic from DNS

One of the challenges with packet captures is that they often contain a mix of traffic from devices, destinations, and applications. When diagnosing performance problems with a particular service, often the first challenge is identifying and extracting the subset of traffic corresponding to that service.

In this exercise, we will use the domain name system lookups to a set of domains that we know are associated with Netflix to identify the IP addresses (and thus, the traffic flows) that are associated with Netflix.

In [2]:
!pip install dnslib

In [3]:
import dnslib

NF_DOMAINS = (["nflxvideo", 
              "netflix", 
              "nflxso", 
              "nflxext"])

### Load the Packet Capture and Identify Netflix Traffic

First, load the traffic capture and inspect it.

In [33]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com
...,...,...,...,...,...,...,...
141466,141467,2018-02-11 08:18:16.299107,192.168.43.72,ec2-52-208-128-101.eu-west-1.compute.amazonaws...,TCP,54,58518 > 443 [RST] Seq=28946 Win=0 Len=0
141467,141468,2018-02-11 08:18:16.299132,192.168.43.72,ec2-52-208-128-101.eu-west-1.compute.amazonaws...,TCP,54,58518 > 443 [RST] Seq=28946 Win=0 Len=0
141468,141469,2018-02-11 08:18:16.299136,192.168.43.72,104.31.113.215,TCP,54,58530 > 80 [ACK] Seq=360 Ack=568 Win=262144 Len=0
141469,141470,2018-02-11 08:18:16.313540,par10s38-in-f3.1e100.net,192.168.43.72,TCP,66,"443 > 58514 [FIN, ACK] Seq=5175 Ack=3726 Win=5..."


#### Filter for DNS Traffic 

Next, write an expression that filters the dataframe to include only DNS traffic.

In [5]:
dns_traffic = ndf[ndf["Protocol"] == "DNS"]
dns_traffic.head()

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com


#### Filter for Netflix DNS Query IDs

Because you are looking for the IP addresses that are associated with Netflix traffic, you need to match the *responses* of the corresponding DNS lookups to the queries that contain Netflix domains. You can link them with the transaction ID in the DNS traffic.

In [13]:
netflix_queries = dns_traffic[dns_traffic["Info"].str.contains("netflix", case=False, na=False)]
netflix_queries

,No.,Time,Source,Destination,Protocol,Length,Info
86,87,2018-02-11 08:10:02.362996,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0xb19a A www.netflix.com
89,90,2018-02-11 08:10:02.363675,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,86,Standard query 0x1f03 A customerevents.netflix...
1010,1011,2018-02-11 08:10:11.797457,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,81,Standard query 0x5542 A push.prod.netflix.com
48478,48479,2018-02-11 08:12:08.379888,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x82dc A www.netflix.com
48571,48572,2018-02-11 08:12:09.390903,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x86ee A www.netflix.com
138767,138768,2018-02-11 08:17:10.645521,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0xed3f A www.netflix.com
138810,138811,2018-02-11 08:17:11.657573,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0xa9b1 A www.netflix.com


#### Find the Associated Netflix IP Addresses

Get the IP addresses associated with all Netflix traffic in the trace.

In [ ]:
# 3) Transaction IDs from those queries
netflix_queries = netflix_queries.copy()
netflix_queries["Transaction_ID"] = netflix_queries["Info"].str.extract(r"(0x[0-9a-fA-F]+)")

# 4) Match responses with same IDs (and ensure they're responses)
txn_ids = netflix_queries["Transaction_ID"].dropna().unique()
pattern = "|".join(txn_ids) if len(txn_ids) else r"$^"  # safe if empty
netflix_responses = dns_traffic[
    dns_traffic["Info"].str.contains(pattern, na=False) &
    dns_traffic["Info"].str.contains("response", case=False, na=False)
].copy()

# 5) Extract IPv4s from responses (A records)
netflix_responses["IP_Address"] = netflix_responses["Info"].str.extract(
    r"(\d{1,3}(?:\.\d{1,3}){3})"
)

# Unique Netflix IPs seen in the trace
netflix_ips = netflix_responses["IP_Address"].dropna().unique().tolist()
netflix_ips


['52.19.39.146',
 '52.210.19.176',
 '34.252.77.54',
 '52.48.148.78',
 '52.48.8.150',
 '52.208.128.101',
 '52.210.133.255']

### Step 2: Counting Traffic to Each Netflix Destination

An important feature for inferring video quality of experience is the throughput of each flow in the video stream. To compute throughput, we need to divide the number of bytes transferred per unit time.

As a first step towards computing that feature, count the number of packets and bytes, in each direction, to each Netflix IP address in the trace.

In [20]:
import re

# Add capturing group
ipv4 = r"(\b(?:\d{1,3}\.){3}\d{1,3}\b)"

ndf = ndf.copy()
ndf["Source_ip"]      = ndf["Source"].astype(str).str.extract(ipv4, expand=False)
ndf["Destination_ip"] = ndf["Destination"].astype(str).str.extract(ipv4, expand=False)


In [30]:
netflix_responses.head(20)


,No.,Time,Source,Destination,Protocol,Length,Info,IP_Address
102,103,2018-02-11 08:10:02.902776,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,545,Standard query response 0xb19a A 52.19.39.146,52.19.39.146
103,104,2018-02-11 08:10:02.902810,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,534,Standard query response 0x1f03 A 52.210.19.176,52.210.19.176
1019,1020,2018-02-11 08:10:12.323453,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,547,Standard query response 0x5542 A 34.252.77.54,34.252.77.54
48593,48594,2018-02-11 08:12:11.299417,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,545,Standard query response 0x82dc A 52.48.148.78,52.48.148.78
48700,48701,2018-02-11 08:12:12.460711,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,545,Standard query response 0x86ee A 52.48.8.150,52.48.8.150
138885,138886,2018-02-11 08:17:13.245854,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,545,Standard query response 0xed3f A 52.208.128.101,52.208.128.101
138938,138939,2018-02-11 08:17:14.622926,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,545,Standard query response 0xa9b1 A 52.210.133.255,52.210.133.255


In [ ]:
import pandas as pd
import re

# 1) Get unique Netflix IPs from your populated responses df
netflix_ips = netflix_responses["IP_Address"].dropna().unique().tolist()

# 2) Extract literal IPv4s for traffic counting
ipv4 = r"(\b(?:\d{1,3}\.){3}\d{1,3}\b)"
ndf = ndf.copy()
ndf["Source_ip"]      = ndf["Source"].astype(str).str.extract(ipv4, expand=False)
ndf["Destination_ip"] = ndf["Destination"].astype(str).str.extract(ipv4, expand=False)

# 3) Set your local client IP (from your DNS queries it’s 192.168.43.72)
local_ip = "192.168.43.72"

# 4) Ignore DNS when counting data traffic
traffic = ndf[ndf["Protocol"] != "DNS"].copy()

# 5) Build direction masks
to_netflix   = (traffic["Source_ip"] == local_ip) & (traffic["Destination_ip"].isin(netflix_ips))
from_netflix = (traffic["Destination_ip"] == local_ip) & (traffic["Source_ip"].isin(netflix_ips))

up_df   = traffic[to_netflix].copy()     # client → Netflix (upstream)
down_df = traffic[from_netflix].copy()   # Netflix → client (downstream)

# 6) Aggregate counts
up_counts = (up_df
    .groupby("Destination_ip", dropna=True)
    .agg(up_pkts=("No.", "count"), up_bytes=("Length", "sum"))
    .rename_axis("Netflix_IP")
    .reset_index())

down_counts = (down_df
    .groupby("Source_ip", dropna=True)
    .agg(down_pkts=("No.", "count"), down_bytes=("Length", "sum"))
    .rename_axis("Netflix_IP")
    .reset_index()
    .rename(columns={"Source_ip": "Netflix_IP"}))

# 7) Combine upstream + downstream per Netflix IP
netflix_flow_stats = (down_counts
    .merge(up_counts, on="Netflix_IP", how="outer")
    .fillna(0)
    .astype({"down_pkts": int, "down_bytes": int, "up_pkts": int, "up_bytes": int})
    .sort_values(["down_bytes","up_bytes"], ascending=False)
    .reset_index(drop=True))

netflix_flow_stats

,Netflix_IP,down_pkts,down_bytes,up_pkts,up_bytes


#### Count the Number of Downstream Bytes and Packets

#### Count the Number of Upstream Bytes and Packets

### Step 3: Inferring Segment Downloads

Another important feature that can be used in inferring video quality of experience is the number of segments per unit time. In this step we will infer the number of segments downloaded per unit time for each IP address.

The number of segments can be determined by counting the number of continuous downstream transfers separated by a packet with a payload of zero bytes. For the last step, compute the number of segment downloads from each Netflix IP address.

## Thought Question

In this exercise, we used the domain name system (DNS) lookup traffic to identify Netflix traffic. This approach can work in practice but is far from perfect, for a number of reasons:

* Domain names can change over time.
* DNS traffic is becoming increasingly encrypted, making it difficult to see domain name lookups and responses.

This turns the problem of *service identification* (i.e., identifying Netflix traffic itself) into an inference/machine learning problem. What features can you think of that could work for developing a model that can perform this type of inference?